# Create dataset 

- Objective: This code joins all the data into a single file which we use for the analysis
- Authors: Joel and Gabriel
- Date: 02-24-2023

## Peliminaries

In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import kWh_consumption as kc

## Import data

In [2]:
# EICV Mathched Data
eicv_reg_data = pd.read_csv('../data/processed/matched_eicv_reg_25_03_23_2008.csv') # matched data (monthly)
eicv_reg_data['matched'] = 1
cols = ['hhid','month_payment','year','month','kWhs','amounts','annual_amount', 'annual_kWhs','avg_month_purchase_count_all_time',
        'avg_month_purchase_count_2016','avg_month_purchase_count_2017','nisr_reg_payment_diff','matched']
eicv_reg_data = eicv_reg_data[cols]
        # rename reg kWh columns to properly identify them as from REG
eicv_reg_data = eicv_reg_data.rename(columns={
    'kWhs':'reg_m_electricity_kWhs',
    'amounts':'reg_m_electricity_payt',
    'month_payment':'nisr_m_electricity_payt',
})
# eicv_reg_data = eicv_reg_data[['hhid','reg_m_electricity_kWhs']] # keep only consumption column

# Demographics
# household = pd.read_csv('../data/processed/household.csv') # original eicv5 file
demographics = pd.read_csv('../data/processed/demographics.csv')
interview_year = demographics[['hhid','interview_year']] # year when interview was conducted

# House characteristics
hh_xteristics = pd.read_csv('../data/processed/hh_xteristics.csv')
hh_xteristics = hh_xteristics.drop(columns=['cluster', 'province', 'district'])
HH_salary_and_business = pd.read_csv('../data/processed/HH_salary_and_business.csv')
socio_econ_metrics = pd.read_csv('../data/processed/socio_econ_metrics.csv')
HH_reliability = pd.read_csv('../data/processed/HH_reliability_closest_line_11_21_23.csv').drop(columns='district')
HH_service_access = pd.read_csv('../data/processed/HH_service_access.csv')
HH_work_stats = pd.read_csv('../data/processed/HH_work_stats.csv')


# Appliances 
appliance_count = pd.read_csv('../data/processed/appliance_count.csv')
non_electric_count = pd.read_csv('../data/processed/non_electric_count.csv')
appliance_presence = pd.read_csv('../data/processed/appliance_presence_HH.csv')

# Loans and savings
HH_credit_savings = pd.read_csv('../data/processed/HH_credit_savings.csv')


# Expenditure
expenditure = pd.read_csv('../data/processed/expenditure_14_03_23_0714.csv')
expenditure = expenditure.drop(columns=['cluster', 'province', 'district'])
rent = pd.read_csv('../data/processed/rent.csv')
rent = rent.drop(columns=['cluster', 'province', 'district'])
poverty = pd.read_csv('../data/processed/poverty.csv')
annual_crop_income = pd.read_csv('../data/processed/HH_crop_agriculture.csv')
food_cash_transfers = pd.read_csv('../data/processed/food_cash_transfers.csv')
        # electricity
electricity = pd.read_csv('../data/processed/electricity.csv')
electricity = electricity.drop(columns=['cluster', 'province', 'district'])
grid_elec = electricity[['hhid','m_electricity_payment','grid']]
grid_elec = grid_elec[(grid_elec.grid==1)]
        # compute monthly electricity consumption in kWh
grid_elec_year = grid_elec.merge(interview_year, how='left')
grid_elec_year['nisr_m_electricity_kWh'] =grid_elec_year.apply(lambda x: kc.electricity_consumption(x.interview_year,x.m_electricity_payment),axis=1)
grid_elec_year = grid_elec_year[['hhid','nisr_m_electricity_kWh']] # downsize dataframe to key columns

# instrument variables
lightning = pd.read_csv('../data/processed/IV_feeder_lightning_metrics.csv')
rainfall  = pd.read_csv('../data/processed/IV_feeder_rainfall_metrics.csv')
poles     = pd.read_csv('../data/processed/IV_feeder_pole_metrics.csv')
krig_IVs  = pd.read_csv('../data/processed/survey_instrument_vars_11_30_23.csv')
hh_grid_dist = pd.read_csv('../data/processed/IV_grid_dist_metrics.csv')

## Merge data

In [12]:
HH_service_access

,hhid,road_unsatisfactory,road_satisfactory,stage_unsatisfactory,stage_satisfactory,time_2_food_market_or_shop,dist_2_food_market_or_shop,time_2_transport_stage,dist_2_transport_stage,time_2_all_weather_roads,...,time_2_health_centre,dist_2_health_centre,time_2_sector_office,dist_2_sector_office,time_2_Cellule_office,dist_2_Cellule_office,bicycle_access,motorcycle_access,vehicle_access,minibus_access
0,200001.0,0,1,0,1,30.0,2.0,30.0,2.0,5.0,...,30.0,2.0,15.0,1.0,7.0,0.30,0,0,0,0
1,200002.0,0,1,0,1,20.0,1.3,15.0,1.0,5.0,...,40.0,2.5,20.0,1.2,7.0,0.50,0,0,1,0
2,200003.0,0,1,0,1,30.0,2.0,15.0,1.0,5.0,...,20.0,1.3,15.0,1.0,7.0,0.50,0,0,0,0
3,200004.0,0,1,0,1,20.0,2.0,10.0,1.0,3.0,...,40.0,2.5,30.0,2.0,10.0,0.80,0,1,0,0
4,200005.0,0,1,1,0,15.0,1.5,10.0,1.0,3.0,...,15.0,1.0,20.0,3.0,5.0,0.40,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14575,214576.0,0,1,0,1,60.0,12.0,90.0,9.0,1.0,...,30.0,3.0,35.0,3.5,2.0,0.20,1,0,0,0
14576,214577.0,0,1,1,0,60.0,12.0,90.0,9.0,1.0,...,30.0,3.0,35.0,3.5,2.0,0.20,1,0,0,0
14577,214578.0,1,0,0,0,60.0,12.0,NaN,NaN,1.0,...,30.0,3.0,35.0,3.5,2.0,0.15,0,0,0,0
14578,214579.0,1,0,1,0,115.0,11.5,90.0,9.0,1.0,...,30.0,3.0,35.0,3.5,1.0,0.10,1,0,0,0


In [33]:
# variables to be used in regression analysis
_ = demographics.merge(expenditure, how='left',on='hhid')
_ = _.merge(non_electric_count, how='left',on='hhid')
_ = _.merge(appliance_count, how='left',on='hhid')
_ = _.merge(appliance_presence, how='left',on='hhid')
_ = _.merge(rent, how='left',on='hhid')
_ = _.merge(electricity, how='left',on='hhid')
_ = _.merge(hh_xteristics, how='left',on='hhid')
_ = _.merge(HH_credit_savings, how='left',on='hhid')
_ = _.merge(poverty, how='left',on='hhid')

_ = _.merge(socio_econ_metrics, how='left',on='hhid')
_ = _.merge(HH_reliability, how='left',on='hhid')
_ = _.merge(HH_salary_and_business, how='left',on='hhid')
_ = _.merge(annual_crop_income, how='left',on='hhid')
_ = _.merge(food_cash_transfers, how='left',on='hhid')
_ = _.merge(HH_service_access, how='left',on='hhid')
_ = _.merge(HH_work_stats, how='left',on='hhid')
_ = _.merge(grid_elec_year, how='left',on='hhid')
_ = _.merge(eicv_reg_data, how='left',on='hhid')
    # merge IVs
_ = _.merge(krig_IVs, how='left',on='hhid')
_ = _.merge(lightning, how='left',on='Feeder_Nam')
_ = _.merge(rainfall, how='left',on='Feeder_Nam')
_ = _.merge(hh_grid_dist, how='left',on='hhid')
data = _.merge(poles, how='left',on='Feeder_Nam')  


# Add zeros to unmatched (these are hhs not matched in the REG dataset)
data['matched'] = data['matched'].fillna(0)


# # Add zeros to amounts (might need to delete)
# data['amounts'] = data['amounts'].fillna(0)
# data['kWhs'] = data['kWhs'].fillna(0)




## Convert annual variable to monthly timeframe

In [34]:
# Define columns
annual_expenses = [
                #annual
                'a_clothing', 'a_belongings','a_housing', 'a_furnishings',
                'a_appliances_subset_of_a_furnishings','a_transport', 'a_leisure', 
                'a_health', 'a_other','rent_total'] 
monthly_expenses = [
                # monthly
                'm_housing','m_transport', 'm_leisure', 'm_care', 'm_communication',
                'm_health','m_other', 
                # monthly too
                'f_leisure', 'f_care', 'f_petrol', 'f_diesel', 'f_transport','f_gas', 
                'f_kerosene', 'f_charcoal', 'f_wood', 'f_energy_materials', 'f_fuel_generator', 
                'f_repairs', 'f_communication', 'f_other', 
                'food',

                # monthly also
                'm_electricity_payment','m_cooking_cost'
                ]

school_costs= [
       'school_registration_fees', 'parent_contributions', 'school_uniforms',
       'school_supplies', 'school_transport', 'school_room_board',
       'other_school_costs', 'total_educ_costs', 'vocational_costs'
            ]

expense_cols  = annual_expenses+monthly_expenses#+school_costs

# Create variables
data_monthly = data.copy()
data_monthly = data_monthly.rename(columns={
    'month_payment':'month_elec_amount_nisr_matched_w_reg',
    'elec_expense':'annual_elec_amount_nisr13',
    'amounts':'month_elec_amounts_reg',
    'kWhs':'month_elec_kwhs_reg',
    'annual_amounts':'annual_elec_amounts_reg',
    'annual_kWhs':'annual_elec_kwhs_reg',
    'annaul_consumption':'annual_expenses_nisr'
})# rename electricity payt cols, differentiate qsel calculation from nisr calculation

# convert annual to monthly expenditure
data_monthly.loc[:,annual_expenses] = data_monthly.loc[:,annual_expenses].div(12)
data_monthly.loc[:,school_costs] = data_monthly.loc[:,school_costs].div(12)

# Monthly expenditure
data_monthly['monthly_expenses_qsel'] = data_monthly.loc[:,expense_cols].sum(axis=1)# annual expenses
data_monthly['monthly_expenses_nisr'] = data_monthly['annual_expenses_nisr']/12 # monthly expenses
data_monthly['log_monthly_expenses_qsel']  = np.log(data_monthly['monthly_expenses_qsel'])
data_monthly['log_monthly_expenses_nisr']  = np.log(data_monthly['monthly_expenses_nisr'])
data_monthly.shape

/home/qsel/miniconda3/envs/geo_env/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


(14580, 354)

In [35]:
district_dict = {'nyarugenge':1,      
 'gasabo':2,
 'kicukiro':3,
 'nyanza':4,
 'gisagara':5,
 'nyaruguru':6,
 'huye':7,
 'nyamagabe':8,
 'ruhango':9,
 'muhanga':10,
 'kamonyi':11,
 'karongi':12,
 'rutsiro':13,
 'rubavu':14,
 'nyabihu':15,
 'ngororero':16,
 'rusizi':17,
 'nyamasheke':18,
 'rulindo':19,
 'gakenke':20,
 'musanze':21,
 'burera':22,
 'gicumbi':23,
 'rwamagana':24,
 'nyagatare':25,
 'gatsibo':26,
 'kayonza':27,
 'kirehe':28,
 'ngoma':29,
 'bugesera':30
 }
provice_dict = {'kigali':1, 'southern':2, 'western':3, 'northern':4, 'eastern':5}
data_monthly['district_str'] = data_monthly.district
data_monthly['province_str'] = data_monthly.province
data_monthly.district = data_monthly.district.replace(district_dict)
data_monthly.province = data_monthly.province.replace(provice_dict)


## Save data

In [38]:
%%time
data_monthly.to_csv('../data/processed/sample_qsel_11_30_23.csv',index=False)

CPU times: user 2.49 s, sys: 135 ms, total: 2.62 s
Wall time: 2min 38s


In [37]:
data_monthly[data_monthly.hhid.duplicated()]

,hhid,interview_month,interview_year,cluster,province,district,urban_rural,people,members,male,...,IV_substation_distance,IV_feeder_pole_means,IV_feeder_pole_medians,IV_feeder_pole_maximums,monthly_expenses_qsel,monthly_expenses_nisr,log_monthly_expenses_qsel,log_monthly_expenses_nisr,district_str,province_str


In [24]:
data = pd.read_csv('../data/processed/sample_qsel_11_21_23.csv')

In [ ]:
data